In [1]:
from transformers import RobertaTokenizer, T5ForConditionalGeneration

tokenizer = RobertaTokenizer.from_pretrained('Salesforce/codet5-small')
model = T5ForConditionalGeneration.from_pretrained('Salesforce/codet5-small')

text = "def greet(user): print(f'hello <extra_id_0>!')"
input_ids = tokenizer(text, return_tensors="pt").input_ids

# simply generate a single sequence
generated_ids = model.generate(input_ids, max_length=10)
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))
# this prints "user: {user.name}"


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/703k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/294k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

user: {user.name}


In [2]:
from transformers import RobertaTokenizer, T5ForConditionalGeneration
from torch.utils.data import Dataset, DataLoader
import torch

# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained('Salesforce/codet5-small')

# Custom Dataset for Verilog to C translation
class VerilogToCCodeDataset(Dataset):
    def __init__(self, verilog_file, c_code_file, tokenizer, max_length=512):
        with open(verilog_file, 'r') as vf:
            self.verilog_lines = vf.readlines()
        with open(c_code_file, 'r') as cf:
            self.c_code_lines = cf.readlines()

        # Ensure both files have the same number of lines
        assert len(self.verilog_lines) == len(self.c_code_lines), "Mismatch in number of Verilog and C code lines"
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.verilog_lines)

    def __getitem__(self, idx):
        verilog_code = self.verilog_lines[idx].strip()
        c_code = self.c_code_lines[idx].strip()

        # Tokenize Verilog (source) and C (target)
        input_encoding = self.tokenizer(
            verilog_code,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        target_encoding = self.tokenizer(
            c_code,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        # Prepare the input and target for training
        return {
            'input_ids': input_encoding['input_ids'].squeeze(),
            'attention_mask': input_encoding['attention_mask'].squeeze(),
            'labels': target_encoding['input_ids'].squeeze()
        }

# Example paths to your dataset
verilog_file = "train1.src"
c_code_file = "train1.tgt"

# Create dataset and dataloader
dataset = VerilogToCCodeDataset(verilog_file, c_code_file, tokenizer)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)


In [3]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, T5ForConditionalGeneration
import torch

# Check GPU availability
print("Is CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("Number of GPUs:", torch.cuda.device_count())

# Load pre-trained CodeT5 model
model = T5ForConditionalGeneration.from_pretrained('Salesforce/codet5-small')

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model is on GPU:", next(model.parameters()).is_cuda)

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    save_total_limit=2,
    save_strategy="epoch",
    predict_with_generate=True,
    # Add verbose logging
    logging_strategy="steps",  # Log at every step
    logging_steps=100,  # Log every 100 steps
    logging_first_step=True,  # Log the first step
    report_to=["none"],  # Disable wandb logging
)

# Trainer for fine-tuning
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    eval_dataset = dataset
)

# Fine-tune the model
trainer.train()

Is CUDA available: True
GPU Name: Tesla T4
Number of GPUs: 1
Model is on GPU: True


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.631000,0.221349
2,0.246300,0.184006
3,0.214200,0.174487


TrainOutput(global_step=507, training_loss=0.33021696008874113, metrics={'train_runtime': 823.4688, 'train_samples_per_second': 4.9, 'train_steps_per_second': 0.616, 'total_flos': 546104168939520.0, 'train_loss': 0.33021696008874113, 'epoch': 3.0})

In [4]:
from transformers import RobertaTokenizer, T5ForConditionalGeneration
from torch.utils.data import Dataset, DataLoader
import torch
from tqdm import tqdm
from nltk.translate.bleu_score import corpus_bleu
import nltk
import csv
nltk.download('punkt')

# Load tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained('Salesforce/codet5-small')
# model = T5ForConditionalGeneration.from_pretrained('./results')  # Load your fine-tuned model

# Custom Dataset for testing
class TestDataset(Dataset):
    def __init__(self, src_file, tgt_file, tokenizer, max_length=512):
        with open(src_file, 'r') as sf, open(tgt_file, 'r') as tf:
            self.src_lines = sf.readlines()
            self.tgt_lines = tf.readlines()
        assert len(self.src_lines) == len(self.tgt_lines), "Mismatch in number of source and target lines"
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.src_lines)

    def __getitem__(self, idx):
        src = self.src_lines[idx].strip()
        tgt = self.tgt_lines[idx].strip()
        return src, tgt

# Load test data
test_dataset = TestDataset('test1.src',
                           'test1.tgt',
                           tokenizer)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Generate predictions
predictions = []
references = []

with torch.no_grad():
    for src, tgt in tqdm(test_loader, desc="Generating predictions"):
        input_ids = tokenizer(src[0], return_tensors="pt", max_length=512, truncation=True, padding="max_length").input_ids.to(device)
        output = model.generate(input_ids, max_length=512, num_beams=4, early_stopping=True)
        decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)
        predictions.append(decoded_output.split())
        references.append([tgt[0].split()])

# Calculate BLEU score
bleu_score = corpus_bleu(references, predictions)
print(f"BLEU Score: {bleu_score}")

# Print some example translations
print("\nExample Translations:")
for i in range(5):
    print(f"\nExample {i + 1}:")
    print(f"Source (Verilog): {test_dataset[i][0]}")
    print(f"Target (C): {test_dataset[i][1]}")
    print(f"Prediction: {' '.join(predictions[i])}")

# Filepath to save the CSV
output_file = "output.csv"

# Writing to the CSV file
with open(output_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    # Writing the header
    writer.writerow(["Example Number", "Source (Verilog)", "Target (C)", "Prediction"])

    # Writing each example
    for i in range(len(test_dataset)):
        source_verilog = test_dataset[i][0]
        target_c = test_dataset[i][1]
        prediction = ' '.join(predictions[i])
        writer.writerow([i + 1, source_verilog, target_c, prediction])

print(f"Output has been written to {output_file}")

# Optional: Save predictions to a file
with open('predictions.txt', 'w') as f:
    for pred in predictions:
        f.write(' '.join(pred) + '\n')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
Generating predictions: 100%|██████████| 30/30 [00:53<00:00,  1.78s/it]

BLEU Score: 0.39118280012708645

Example Translations:

Example 1:
Source (Verilog): module entry_point ( , ap_clk, , ap_rst, , ap_start, , ap_done, , ap_idle, , ap_ready, , n, , item, , temptemp1, , temptemp1_ap_vld, , temptemp2, , temptemp2_ap_vld, , temptemp3, , temptemp3_ap_vld, , ap_return , ); , parameter ap_ST_fsm_state1 = 3'd1; , parameter ap_ST_fsm_state2 = 3'd2; , parameter ap_ST_fsm_state3 = 3'd4; , input ap_clk; , input ap_rst; , input ap_start; , output ap_done; , output ap_idle; , output ap_ready; , input [31:0] n; , input [31:0] item; , output [31:0] temptemp1; , output temptemp1_ap_vld; , output [31:0] temptemp2; , output temptemp2_ap_vld; , output [31:0] temptemp3; , output temptemp3_ap_vld; , output [31:0] ap_return; , reg ap_done; , reg ap_idle; , reg ap_ready; , reg temptemp1_ap_vld; , reg temptemp2_ap_vld; , reg temptemp3_ap_vld; , (* fsm_encoding = "none" *) reg [2:0] ap_CS_fsm; , wire ap_CS_fsm_state1; , wire [30:0] trunc_ln29_fu_90_p1; , reg [30:0] trunc_ln29_re